In [1]:
library(tidyverse)
library(limma)

Warning message in system("timedatectl", intern = TRUE):
“running command 'timedatectl' had status 1”
── Attaching packages ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 1.3.1 ──

✔ ggplot2 3.3.5     ✔ purrr   0.3.4
✔ tibble  3.1.5     ✔ dplyr   1.0.7
✔ tidyr   1.1.4     ✔ stringr 1.4.0
✔ readr   2.0.2     ✔ forcats 0.5.1

── Conflicts ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()



In [2]:
counts_df <- read_tsv("/mnt/d/gene_expression_example_data/microarray_endometriosis_counts.tsv")
coldata_df <- read_tsv("/mnt/d/gene_expression_example_data/microarray_endometriosis_coldata.tsv")
coldata_df <- coldata_df %>%
    mutate(condition = factor(condition, levels = c("normal", "endometriosis")))

Rows: 21407 Columns: 154

── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr   (1): symbol
dbl (153): GSM109815, GSM109816, GSM109817, GSM109828, GSM109829, GSM109830,...


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

Rows: 153 Columns: 8

── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (7): sample_name, series, title, condition, endometriosis_stage, phase, ...
dbl (1): batch


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.



In [3]:
min_expr <- log2(50)
min_prop <- 0.25

In [4]:
counts_df

symbol,GSM109815,GSM109816,GSM109817,GSM109828,GSM109829,GSM109830,GSM109831,GSM150190,GSM150191,⋯,GSM1256786,GSM1256787,GSM1256788,GSM1256791,GSM1256793,GSM1256796,GSM1256797,GSM1256798,GSM1256799,GSM1256800
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,5.685566,6.309144,6.558803,5.676761,5.713494,5.651049,6.095096,6.734897,6.311050,⋯,5.807815,5.799219,6.019293,5.726396,5.767185,5.945817,5.736468,5.966595,5.868001,5.931074
A1BG-AS1,6.354643,7.025457,7.019929,6.820971,6.986295,7.019873,6.614798,7.162332,7.692469,⋯,6.700733,6.763380,6.717756,6.878098,6.691541,6.742213,6.825399,6.669743,6.579118,6.651996
A1CF,6.017202,6.806242,5.876322,6.075240,6.048896,6.123009,6.480756,6.343271,6.492033,⋯,6.245288,6.031817,5.796515,6.262572,6.067918,6.291459,5.939247,6.274130,6.093899,5.928299
A2M,10.705147,10.982733,11.381476,11.434625,11.319421,11.669190,11.679939,11.021458,10.674658,⋯,11.147112,11.202096,11.563820,11.578159,11.268654,11.595849,11.414081,11.245570,11.330142,11.243454
A2M-AS1,6.193356,6.496668,6.377511,6.072465,5.664001,6.158695,5.950787,6.176829,5.792399,⋯,6.171834,6.317519,6.506197,5.722692,6.196140,6.186685,6.062160,6.451924,6.235170,5.969264
A2ML1,5.936272,6.090145,5.661428,5.772800,5.649532,5.542943,6.006635,5.752719,6.293694,⋯,5.671488,5.636839,5.811221,5.782499,5.628093,5.671767,5.643708,5.561749,5.711164,5.600078
A2MP1,6.193356,6.532157,5.972719,6.051718,6.131620,6.140529,6.239424,6.046551,6.383859,⋯,6.038329,6.151578,6.124441,6.200450,6.225230,6.266043,5.956293,6.194836,6.257100,6.103579
A4GALT,7.399494,7.314846,7.260542,6.702430,7.204924,7.250284,7.084704,7.514505,7.745656,⋯,7.101046,6.920610,6.783544,7.082056,7.143511,7.054332,6.971775,6.935188,7.100574,7.065269
A4GNT,6.212988,6.594580,6.254428,6.024918,6.118253,6.123767,6.236670,6.186598,6.752778,⋯,6.290891,6.244271,6.052436,6.181848,6.126513,6.163117,6.266918,6.129302,6.278243,6.216228


In [5]:
filt_counts_df <- counts_df %>%
    filter(rowSums(.[-1] > min_expr) / (ncol(.) - 1) >= min_prop)
filt_expr <- filt_counts_df %>%
    column_to_rownames("symbol") %>%
    as.matrix()

In [6]:
filt_expr

,GSM109815,GSM109816,GSM109817,GSM109828,GSM109829,GSM109830,GSM109831,GSM150190,GSM150191,GSM150192,⋯,GSM1256786,GSM1256787,GSM1256788,GSM1256791,GSM1256793,GSM1256796,GSM1256797,GSM1256798,GSM1256799,GSM1256800
A1BG,5.685566,6.309144,6.558803,5.676761,5.713494,5.651049,6.095096,6.734897,6.311050,6.201764,⋯,5.807815,5.799219,6.019293,5.726396,5.767185,5.945817,5.736468,5.966595,5.868001,5.931074
A1BG-AS1,6.354643,7.025457,7.019929,6.820971,6.986295,7.019873,6.614798,7.162332,7.692469,7.116755,⋯,6.700733,6.763380,6.717756,6.878098,6.691541,6.742213,6.825399,6.669743,6.579118,6.651996
A1CF,6.017202,6.806242,5.876322,6.075240,6.048896,6.123009,6.480756,6.343271,6.492033,6.303324,⋯,6.245288,6.031817,5.796515,6.262572,6.067918,6.291459,5.939247,6.274130,6.093899,5.928299
A2M,10.705147,10.982733,11.381476,11.434625,11.319421,11.669190,11.679939,11.021458,10.674658,11.074659,⋯,11.147112,11.202096,11.563820,11.578159,11.268654,11.595849,11.414081,11.245570,11.330142,11.243454
A2M-AS1,6.193356,6.496668,6.377511,6.072465,5.664001,6.158695,5.950787,6.176829,5.792399,6.005239,⋯,6.171834,6.317519,6.506197,5.722692,6.196140,6.186685,6.062160,6.451924,6.235170,5.969264
A2ML1,5.936272,6.090145,5.661428,5.772800,5.649532,5.542943,6.006635,5.752719,6.293694,5.755191,⋯,5.671488,5.636839,5.811221,5.782499,5.628093,5.671767,5.643708,5.561749,5.711164,5.600078
A2MP1,6.193356,6.532157,5.972719,6.051718,6.131620,6.140529,6.239424,6.046551,6.383859,6.206954,⋯,6.038329,6.151578,6.124441,6.200450,6.225230,6.266043,5.956293,6.194836,6.257100,6.103579
A4GALT,7.399494,7.314846,7.260542,6.702430,7.204924,7.250284,7.084704,7.514505,7.745656,7.672225,⋯,7.101046,6.920610,6.783544,7.082056,7.143511,7.054332,6.971775,6.935188,7.100574,7.065269
A4GNT,6.212988,6.594580,6.254428,6.024918,6.118253,6.123767,6.236670,6.186598,6.752778,6.268343,⋯,6.290891,6.244271,6.052436,6.181848,6.126513,6.163117,6.266918,6.129302,6.278243,6.216228
AA06,5.930863,6.421702,5.644567,5.882695,5.635277,5.650887,6.069808,6.140968,6.651462,6.054905,⋯,5.617088,5.788808,5.698525,5.776924,5.636527,5.675551,5.688569,5.665631,5.729384,5.633397


In [7]:
design <- model.matrix(~ condition, data = coldata_df)
rownames(design) <- coldata_df$sample_name

In [8]:
design
all(colnames(filt_expr) == rownames(design))

,(Intercept),conditionendometriosis
GSM109815,1,0
GSM109816,1,0
GSM109817,1,0
GSM109828,1,0
GSM109829,1,0
GSM109830,1,0
GSM109831,1,0
GSM150190,1,1
GSM150191,1,1
GSM150192,1,1


[1] TRUE

In [9]:
qual_weights <- arrayWeights(filt_expr, design = design)

In [10]:
lm_fit <- lmFit(filt_expr, design = design, weights = qual_weights)
bayes_fit <- eBayes(lm_fit)

In [11]:
bayes_fit$coefficients %>% colnames()

[1] "(Intercept)"            "conditionendometriosis"

In [12]:
fit_de_res_df <- topTable(bayes_fit, coef = "conditionendometriosis", number = nrow(filt_counts_df), adjust.method = "BH") %>%
    rename(lfc = logFC, ave_expr = AveExpr, pval = P.Value, padj = adj.P.Val) %>%
    as_tibble(rownames = "symbol")

In [15]:
2^(0.9797001)

[1] 1.972055

In [13]:
fit_de_res_df

symbol,lfc,ave_expr,t,pval,padj,B
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
RBM25,-0.9797001,9.271442,-115.68463,5.281688e-151,9.067602e-147,334.9097
PRH1,0.5228194,6.565680,99.84267,2.477561e-141,2.126738e-137,312.9926
PLPP6,-0.8366814,6.691173,-91.11445,2.401855e-135,1.374502e-131,299.3625
ZBED8,-0.9876665,5.984509,-88.96926,8.642004e-134,3.709148e-130,295.8130
ZMAT3,-0.7141661,7.014628,-88.67287,1.427071e-133,4.878826e-130,295.3159
FECH,-0.5544542,7.576011,-88.56792,1.705088e-133,4.878826e-130,295.1395
FOSB,2.3816607,8.079308,87.88804,5.428193e-133,1.331303e-129,293.9916
XRN2,-0.8461322,8.257239,-86.54415,5.493155e-132,1.178831e-128,291.6966
CWC15,-0.6416429,9.325332,-85.44062,3.771230e-131,7.193831e-128,289.7855
